## Studi Kasus 2: Transisi Fasa dan Kritikalitas pada T = 2.27

### Model dan Hamiltonian

Model Ising 2D merepresentasikan sistem spin pada kisi N×N, di mana setiap
spin s_i ∈ {−1, +1} berinteraksi hanya dengan empat tetangga terdekatnya.
Energi sistem diberikan oleh Hamiltonian:

H = −J Σ⟨i,j⟩ s_i s_j

dengan J adalah konstanta kopling (J = 1, ferromagnetik) dan ⟨i,j⟩ menyatakan
pasangan spin tetangga terdekat. Pada simulasi ini, kondisi batas periodik
(periodic boundary conditions) diterapkan agar tidak ada efek tepi (edge
effect) yang mendistorsi perilaku sistem ukuran-hingga (finite-size).

### Suhu Kritis Teoritis

Solusi eksak Onsager (1944) untuk kisi persegi 2D tanpa medan magnet luar
memberikan suhu kritis:

T_c = 2 / ln(1 + √2) ≈ 2.269

Pada T = T_c, sistem mengalami transisi fasa orde kedua (continuous phase
transition) dari fase paramagnetik (T > T_c, spin acak, M ≈ 0) ke fase
ferromagnetik (T < T_c, spin tersusun, |M| → 1). Studi kasus ini menguji
perilaku sistem tepat di titik transisi tersebut, T = 2.27, yang merupakan
titik paling menarik secara fisis karena di sinilah fenomena kritis—
fluktuasi tak terbatas, panjang korelasi divergen, dan hilangnya skala
karakteristik—muncul secara penuh.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

def hitung_energi_total(grid):
    """
    Menghitung energi total awal kisi dengan kondisi batas periodik.
    Menggunakan vektorisasi np.roll agar komputasi lebih cepat.
    """
    tetangga_atas = np.roll(grid, 1, axis=0)
    tetangga_bawah = np.roll(grid, -1, axis=0)
    tetangga_kiri = np.roll(grid, 1, axis=1)
    tetangga_kanan = np.roll(grid, -1, axis=1)

    jumlah_tetangga = tetangga_atas + tetangga_bawah + tetangga_kiri + tetangga_kanan

    # Dibagi 2 untuk mencegah perhitungan ganda antar pasangan tetangga
    energi_total = -np.sum(grid * jumlah_tetangga) / 2.0
    return energi_total

def metropolis_step(grid, T):
    """
    Satu langkah Metropolis yang mengembalikan matriks grid terbaru
    dan perubahan energi (jika diterima).
    """
    N = grid.shape[0]

    # Memilih koordinat spin secara acak
    x, y = random.randint(0, N-1), random.randint(0, N-1)

    # Menghitung jumlah spin tetangga (batas periodik modulo)
    s_neighbors = (
        grid[(x+1)%N, y] + grid[(x-1)%N, y] +
        grid[x, (y+1)%N] + grid[x, (y-1)%N]
    )

    # Menghitung potensi perubahan energi (Delta E)
    delta_E = 2 * grid[x, y] * s_neighbors
    dE_diterima = 0

    # Kriteria Metropolis
    if delta_E <= 0 or random.random() < np.exp(-delta_E / T):
        grid[x, y] *= -1 # Balikkan spin
        dE_diterima = delta_E # Catat perubahan energi yang masuk ke sistem

    return grid, dE_diterima

def run_simulation(N=20, temp=1.0, n_steps=200000):
    """
    Menjalankan simulasi dan merekam makrostate magnetisasi & energi secara efisien.
    """
    # Inisialisasi awal (Hot Start)
    grid = np.random.choice([-1, 1], size=(N, N))
    energi_sistem = hitung_energi_total(grid)

    magnetization_history = []
    energy_history = []

    for step in range(n_steps):
        grid, dE_diterima = metropolis_step(grid, temp)
        energi_sistem += dE_diterima # Update energi hanya berdasarkan dE

        # Pengambilan sampel setiap 100 langkah
        if step % 100 == 0:
            magnetization_history.append(np.mean(grid))
            energy_history.append(energi_sistem / (N**2)) # Energi rata-rata per spin

    return grid, magnetization_history, energy_history

def plot_studi_kasus(grid_size, T, n_steps):
    """
    Memproses satu kasus suhu, menghitung fluktuasi, dan menampilkan 3 plot berdampingan.
    """
    print(f"Menjalankan simulasi untuk T = {T}...")
    final_grid, M_history, E_history = run_simulation(N=grid_size, temp=T, n_steps=n_steps)

    # Konversi ke array NumPy untuk memudahkan perhitungan statistik
    M_array = np.array(M_history)
    E_array = np.array(E_history)

    # Menghitung Fluktuasi menggunakan variansi
    suseptibilitas = np.var(M_array) / T
    kapasitas_kalor = np.var(E_array) / (T**2)

    # Setup Visualisasi
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f'Analisis Termodinamika Sistem pada Suhu T = {T}', fontsize=16, fontweight='bold')

    # Plot 1: Keadaan Akhir Kisi (Domain Magnetik)
    ax_grid = axes[0]
    ax_grid.imshow(final_grid, cmap='binary', vmin=-1, vmax=1)
    ax_grid.set_title("Keadaan Akhir Kisi Spin")
    ax_grid.axis('off')

    # Plot 2: Riwayat Magnetisasi
    ax_mag = axes[1]
    ax_mag.plot(M_array, color='blue', alpha=0.7)
    ax_mag.set_title("Riwayat Magnetisasi Rata-rata")
    ax_mag.set_xlabel("Langkah Monte Carlo (x100)")
    ax_mag.set_ylabel("Magnetisasi (M)")
    ax_mag.set_ylim(-1.1, 1.1)
    ax_mag.grid(True, linestyle='--', alpha=0.5)

    # Plot 3: Riwayat Energi
    ax_e = axes[2]
    ax_e.plot(E_array, color='red', alpha=0.7)
    ax_e.set_title("Riwayat Energi Lokal Rata-rata")
    ax_e.set_xlabel("Langkah Monte Carlo (x100)")
    ax_e.set_ylabel("Energi (E)")
    ax_e.grid(True, linestyle='--', alpha=0.5)

    # Menempelkan nilai statistik di dalam plot energi sebagai legenda
    teks_statistik = (
        f'|M| rata-rata = {abs(np.mean(M_array)):.4f}\n'
        f'E rata-rata = {np.mean(E_array):.4f}\n'
        f'Suseptibilitas (\u03c7) = {suseptibilitas:.4f}\n'
        f'Kapasitas Kalor (Cv) = {kapasitas_kalor:.4f}'
    )
    teks_legenda = mpatches.Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='none')
    ax_e.legend([teks_legenda], [teks_statistik], loc='best', handlelength=0, fontsize=10)

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Parameter Utama Simulasi
    grid_size = 20
    monte_carlo_steps = 200000

    # Daftar Suhu untuk 3 Studi Kasus (Feromagnetik, Kritis, Paramagnetik)
    suhu_studi_kasus = [2.27]

    # Eksekusi berurutan untuk setiap studi kasus
    for suhu in suhu_studi_kasus:
        plot_studi_kasus(grid_size, suhu, monte_carlo_steps)

### Analisis Pola Domain pada Kisi Akhir

Visualisasi kisi akhir menunjukkan ciri khas perilaku kritis: domain hitam
dan putih tidak membentuk satu cluster dominan tunggal (seperti yang akan
terlihat pada T < T_c), maupun tersebar acak tanpa pola (seperti pada
T ≫ T_c). Sebaliknya, domain tersusun dalam struktur bersarang dari
berbagai ukuran—cluster besar mengandung sub-cluster kecil, yang pada
gilirannya mengandung cluster lebih kecil lagi.

Pola hierarkis multi-skala ini adalah tanda tangan visual dari **geometri
fraktal** yang muncul ketika panjang korelasi ξ(T) mendekati divergensi.
Secara teoritis, dekat T_c:

ξ(T) ∝ |T − T_c|^(−ν)

dengan eksponen kritis ν = 1 untuk kelas universalitas Ising 2D. Pada
T = T_c persis, ξ → ∞ untuk sistem tak terbatas, yang berarti korelasi
antar-spin tidak lagi meluruh secara eksponensial terhadap jarak,
melainkan secara power-law. Inilah yang dimaksud dengan **runtuhnya
keteraturan jarak jauh (breakdown of long-range order)**: sistem tidak
memiliki satu skala panjang yang dominan, sehingga pola domain tampak
serupa pada berbagai tingkat pembesaran (self-similarity)—properti
definitif dari fraktal.

Pada kisi berhingga N = 20, ξ tentu dibatasi oleh ukuran sistem itu sendiri
(finite-size cutoff), sehingga divergensi sejati tidak teramati secara
penuh. Namun pola domain yang dihasilkan tetap menunjukkan jejak kuat
dari perilaku kritis ini dibandingkan kisi pada T jauh dari T_c.

In [ ]:
### Critical Slowing Down pada Riwayat Magnetisasi dan Energi

Grafik riwayat magnetisasi menunjukkan bahwa sistem tidak pernah benar-benar
mencapai nilai M yang stabil dan konstan sepanjang 200.000 langkah Monte
Carlo (2000 sampel)—nilai M terus berfluktuasi dengan amplitudo besar dan
bahkan menunjukkan tren naik-turun jangka panjang, alih-alih berosilasi
mantap di sekitar satu nilai kesetimbangan.

Perilaku ini **bukan indikasi kegagalan simulasi**, melainkan manifestasi
langsung dari fenomena fisis bernama **critical slowing down**. Waktu
relaksasi sistem (τ), yaitu waktu yang dibutuhkan untuk "melupakan" kondisi
awalnya, berskala dengan panjang korelasi melalui eksponen kritis dinamis z:

τ ∝ ξ^z

Karena ξ → ∞ pada T_c, maka τ juga divergen. Akibatnya, algoritma Metropolis
single-spin-flip—yang membalik satu spin per langkah—menjadi sangat
tidak efisien dalam mendekorelasi konfigurasi sistem di dekat titik kritis.
Setiap pembalikan spin lokal membutuhkan waktu jauh lebih lama untuk
"merambat" pengaruhnya ke seluruh kisi dibandingkan pada suhu jauh dari T_c.

Hal ini juga konsisten dengan riwayat energi: fluktuasi E tetap besar
sepanjang simulasi tanpa menyempit, mengindikasikan sistem terus
menjelajahi ruang konfigurasi yang luas alih-alih menetap di satu wilayah
energi minimum yang sempit.

In [ ]:
### Interpretasi Suseptibilitas dan Kapasitas Kalor

Dari hasil simulasi diperoleh:
- Suseptibilitas magnetik χ = 0.0159
- Kapasitas kalor Cv = 0.0048

Kedua besaran ini dihitung dari fluktuasi makrostate:

χ = Var(M) / T,        Cv = Var(E) / T²

Secara teoritis, pada sistem **tak terbatas** (thermodynamic limit, N → ∞),
χ dan Cv seharusnya **divergen** di T = T_c—inilah definisi fundamental dari
transisi fasa orde kedua, di mana fluktuasi sistem menjadi tak terbatas.

Nilai χ dan Cv yang relatif kecil dan terhingga pada hasil di atas adalah
konsekuensi dari **efek ukuran-hingga (finite-size effect)**. Karena kisi
hanya berukuran N = 20 (400 spin), panjang korelasi ξ dibatasi secara
artifisial oleh dimensi kisi itu sendiri (ξ_max ≈ N), sehingga divergensi
sejati "terpotong" (cut off). Hubungan ini diformalkan dalam teori
finite-size scaling:

χ_max(N) ∝ N^(γ/ν)

dengan γ = 7/4 untuk kelas universalitas Ising 2D. Artinya, jika simulasi
ini diulang dengan N yang lebih besar (40, 80, 160, ...), nilai puncak χ
pada T ≈ T_c akan meningkat secara sistematis mengikuti hukum pangkat
tersebut—sebuah prediksi yang dapat diverifikasi sebagai pengembangan lebih
lanjut dari studi kasus ini.

In [ ]:
### Kesimpulan Studi Kasus 2

Simulasi Monte Carlo pada T = 2.27 ≈ T_c menunjukkan tiga ciri kunci dari
perilaku kritis pada model Ising 2D:

1. **Domain fraktal**: pola spin pada kisi akhir menunjukkan struktur
   bersarang multi-skala, mengindikasikan panjang korelasi ξ yang mendekati
   ukuran sistem itu sendiri.
2. **Critical slowing down**: riwayat magnetisasi dan energi gagal mencapai
   kestabilan dalam 200.000 langkah, mencerminkan waktu relaksasi τ ∝ ξ^z
   yang divergen di titik kritis.
3. **Fluktuasi kritis terpotong**: nilai χ dan Cv yang terhingga (bukan
   divergen) adalah artefak dari ukuran sistem N = 20 yang terbatas,
   konsisten dengan prediksi finite-size scaling.

Ketiga temuan ini secara koheren menggambarkan bahwa sistem berada tepat
pada ambang transisi fasa orde kedua, di mana tidak ada satu skala panjang
atau waktu yang mendominasi perilaku sistem—berbeda secara kualitatif dari
fase ferromagnetik (T < T_c) yang tertata maupun fase paramagnetik (T > T_c)
yang acak dan tak berkorelasi.